# 2606-Data Ecosystems and Governance in Organizations

# 0. Load Data

In [188]:
from pymongo import MongoClient
from collections import Counter
import pandas as pd
import numpy as np
import json
import re

with open('data/raw_credit_applications.json') as f:
    data = json.load(f)

print("Total records in raw file: ", len(data))

Total records in raw file:  502


In [189]:
# Show all unique fields across dataset
all_keys = set()
for r in data:
    for top_key, value in r.items():
        all_keys.add(top_key)
        if isinstance(value, dict):
            for sub_key in value.keys():
                all_keys.add(f"{top_key}.{sub_key}")
        elif isinstance(value, list) and len(value) > 0:
            if isinstance(value[0], dict):
                for sub_key in value[0].keys():
                    all_keys.add(f"{top_key}[].{sub_key}")

print("All fields found across dataset:")
for key in sorted(all_keys):
    print(f"{key}")

All fields found across dataset:
_id
applicant_info
applicant_info.date_of_birth
applicant_info.email
applicant_info.full_name
applicant_info.gender
applicant_info.ip_address
applicant_info.ssn
applicant_info.zip_code
decision
decision.approved_amount
decision.interest_rate
decision.loan_approved
decision.rejection_reason
financials
financials.annual_income
financials.annual_salary
financials.credit_history_months
financials.debt_to_income
financials.savings_balance
loan_purpose
notes
processing_timestamp
spending_behavior
spending_behavior[].amount
spending_behavior[].category


In [190]:
# Connect to MongoDB and upsert records
client = MongoClient()
db = client['novacred']
collection = db['applications']
collection.drop()

for record in data:
    collection.replace_one({'_id': record['_id']}, record, upsert=True)

print(f"Records in MongoDB after upsert: {collection.count_documents({})}")

Records in MongoDB after upsert: 500


# 1. Duplicates

In [191]:
# Find duplicates
ids = [record['_id'] for record in data]
id_counts = Counter(ids)
duplicate_ids = {id_: count for id_, count in id_counts.items() if count > 1}

print(f"Total records in file : {len(data)}")
print(f"Unique _id values: {len(set(ids))}")
print(f"Duplicate IDs: {list(duplicate_ids.keys())}")

# Show duplicates
for duplicate in duplicate_ids:
    versions = [record for record in data if record['_id'] == duplicate]
    print(f"\n--- Versions of {duplicate} ---")
    for i, v in enumerate(versions):
        print(f"  Version {i+1}: notes='{v.get('notes', 'none')}', keys={sorted(v.keys())}")

Total records in file : 502
Unique _id values: 500
Duplicate IDs: ['app_042', 'app_001']

--- Versions of app_042 ---
  Version 1: notes='none', keys=['_id', 'applicant_info', 'decision', 'financials', 'spending_behavior']
  Version 2: notes='RESUBMISSION', keys=['_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior']

--- Versions of app_001 ---
  Version 1: notes='none', keys=['_id', 'applicant_info', 'decision', 'financials', 'spending_behavior']
  Version 2: notes='DUPLICATE_ENTRY_ERROR', keys=['_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior']


**Notes**:

- The raw dataset contained 502 records, of which 500 were unique applications. app_042 was flagged as a resubmission, and app_001 was flagged as a duplicate entry error. For app_042, we'll keep the resubmission as the most recent version to honor the applicant's latest intent.The duplicate entry error of app_001 will be resolved by restoring the original record.

In [192]:
# SSNs as unique IDs. (different IDs, same person?)
ssns = [(record['_id'], record['applicant_info'].get('ssn')) for record in data]
ssn_counts = Counter(ssn for _, ssn in ssns if ssn)
dup_ssns = {ssn: count for ssn, count in ssn_counts.items() if count > 1}

print(f"Duplicate SSNs (same SSN across different app IDs): {len(dup_ssns)}")
if dup_ssns:
    for ssn, count in dup_ssns.items():
        app_ids = [id_ for id_, s in ssns if s == ssn]
        print(f"SSN {ssn} appears in: {app_ids}")

Duplicate SSNs (same SSN across different app IDs): 3
SSN 652-70-5530 appears in: ['app_042', 'app_042']
SSN 937-72-8731 appears in: ['app_101', 'app_234']
SSN 780-24-9300 appears in: ['app_088', 'app_016']


In [193]:
# Look at duplicate applications (app_042 already resolved)
from pprint import pprint
print("Duplicate Case 1: \n")
pprint(db.applications.find_one({'_id': 'app_101'}))
pprint(db.applications.find_one({'_id': 'app_234'}))
print('-' * 50)
print("\nDuplicate Case 2: \n")
pprint(db.applications.find_one({'_id': 'app_088'}))
pprint(db.applications.find_one({'_id': 'app_016'}))

Duplicate Case 1: 

{'_id': 'app_101',
 'applicant_info': {'date_of_birth': '1997-03-23',
                    'email': 'sandra.smith99@icloud.com',
                    'full_name': 'Sandra Smith',
                    'gender': 'Female',
                    'ip_address': '172.30.237.98',
                    'ssn': '937-72-8731',
                    'zip_code': '90255'},
 'decision': {'loan_approved': False,
              'rejection_reason': 'algorithm_risk_score'},
 'financials': {'annual_income': 55000,
                'credit_history_months': 0,
                'debt_to_income': 0.3,
                'savings_balance': 15364},
 'spending_behavior': [{'amount': 527, 'category': 'Groceries'},
                       {'amount': 464, 'category': 'Entertainment'}]}
{'_id': 'app_234',
 'applicant_info': {'date_of_birth': '1976/01/29',
                    'email': 'samuel.hill67@protonmail.com',
                    'full_name': 'Samuel Hill',
                    'gender': 'Male',
             

**Notes**:
Both SSN duplicates involve records with clearly different names, 
genders, dates of birth and income levels, indicating a data entry 
error rather than the same person applying twice. Both records are 
flagged with 'duplicate_ssn' for human review. The correct SSN 
cannot be determined programmatically and requires manual verification 
against source documents.

## 1.1 Handle Duplicates

In [194]:
# Fix duplicte entry of app_001, keep correct version
# app_001 — restore original, discard DUPLICATE_ENTRY_ERROR version
original_app_001 = next(record for record in data 
                        if record['_id'] == 'app_001' 
                        and 'DUPLICATE_ENTRY_ERROR' not in str(record.get('notes', '')))

# Change to correct version
collection.replace_one({'_id': 'app_001'}, original_app_001, upsert=True)

print(f"\nFinal document count in MongoDB: {collection.count_documents({})}")


Final document count in MongoDB: 500


In [195]:
# Fix SSN duplicates
unique_data = {record['_id']: record for record in data}
ssns = [(id_, record['applicant_info'].get('ssn')) 
        for id_, record in unique_data.items()]

ssn_counts = Counter(ssn for _, ssn in ssns if ssn)
dup_ssns = {ssn: count for ssn, count in ssn_counts.items() if count > 1}

for ssn, count in dup_ssns.items():
    app_ids = [id_ for id_, s in ssns if s == ssn]
    print(f"SSN {ssn} appears in: {app_ids}")

# Flag in MongoDB for human review
for ssn in dup_ssns:
    app_ids = [id_ for id_, s in ssns if s == ssn]
    for app_id in app_ids:
        collection.update_one(
            {'_id': app_id},
            {'$set': {'data_quality_flag': 'duplicate_ssn'}}
        )
        print(f"Flagged {app_id} with 'duplicate_ssn'")

SSN 937-72-8731 appears in: ['app_101', 'app_234']
SSN 780-24-9300 appears in: ['app_088', 'app_016']
Flagged app_101 with 'duplicate_ssn'
Flagged app_234 with 'duplicate_ssn'
Flagged app_088 with 'duplicate_ssn'
Flagged app_016 with 'duplicate_ssn'


# 2. Data Type Check

In [196]:
# Check data types in raw json
fields_to_check = {
    # Financials (numeric)
    'financials.annual_income'        : ('financials', 'annual_income',         (int, float)),
    'financials.credit_history_months': ('financials', 'credit_history_months', (int,)),
    'financials.debt_to_income'       : ('financials', 'debt_to_income',        (int, float)),
    'financials.savings_balance'      : ('financials', 'savings_balance',       (int, float)),
    # Applicant info (all strings)
    'applicant_info.full_name'        : ('applicant_info', 'full_name',         (str,)),
    'applicant_info.email'            : ('applicant_info', 'email',             (str,)),
    'applicant_info.ssn'              : ('applicant_info', 'ssn',               (str,)),
    'applicant_info.ip_address'       : ('applicant_info', 'ip_address',        (str,)),
    'applicant_info.gender'           : ('applicant_info', 'gender',            (str,)),
    'applicant_info.date_of_birth'    : ('applicant_info', 'date_of_birth',     (str,)),
    'applicant_info.zip_code'         : ('applicant_info', 'zip_code',          (str,)),
    # Decision
    'decision.loan_approved'          : ('decision', 'loan_approved',           (bool,)),
    'decision.interest_rate'          : ('decision', 'interest_rate',           (int, float)),
    'decision.approved_amount'        : ('decision', 'approved_amount',         (int, float)),
    'decision.rejection_reason'       : ('decision', 'rejection_reason',        (str,)),
}

print(f"{'Field':<40} {'Data Types Found':<30}")
print("-" * 70)
for label, (section, field, expected_types) in fields_to_check.items():
    type_counts = Counter(
        type(r.get(section, {}).get(field)).__name__
        for r in data
        if r.get(section, {}).get(field) is not None
    )
    print(f"{label:<40} {str(dict(type_counts)):<30}")

# Check spending_behavior separately
spending_amount_issues = [
    record['_id'] for record in data
    for item in record.get('spending_behavior', [])
    if not isinstance(item.get('amount'), (int, float))
]

spending_category_issues = [
    record['_id'] for record in data
    for item in record.get('spending_behavior', [])
    if not isinstance(item.get('category'), str)
]

print(f"spending_behavior.amount: {len(spending_amount_issues)} non-numeric records")
print(f"spending_behavior.category: {len(spending_category_issues)} non-string records")

Field                                    Data Types Found              
----------------------------------------------------------------------
financials.annual_income                 {'int': 488, 'str': 8, 'float': 1}
financials.credit_history_months         {'int': 502}                  
financials.debt_to_income                {'float': 502}                
financials.savings_balance               {'int': 502}                  
applicant_info.full_name                 {'str': 502}                  
applicant_info.email                     {'str': 502}                  
applicant_info.ssn                       {'str': 497}                  
applicant_info.ip_address                {'str': 497}                  
applicant_info.gender                    {'str': 501}                  
applicant_info.date_of_birth             {'str': 501}                  
applicant_info.zip_code                  {'str': 501}                  
decision.loan_approved                   {'bool': 502}       

In [210]:
# Fix annual_income type issues in MongoDB
# Identify records where annual_income is a string instead of a number
type_issues = [
    (record['_id'], record['financials'].get('annual_income'))
    for record in data
    if isinstance(record.get('financials', {}).get('annual_income'), str)
]

# Fix in MongoDB
for app_id, val in type_issues:
    cleaned = int(float(str(val).replace(',', '').strip()))
    collection.update_one(
        {'_id': app_id},
        {'$set': {'financials.annual_income': int(float(str(val)))}}
    )

# 3. Missing and Incomplete Data

In [198]:
# Check in raw data first, field by field

applicant_fields = ['full_name', 'email', 'ssn', 'ip_address',
                    'gender', 'date_of_birth', 'zip_code']

financial_fields = ['annual_income', 'credit_history_months',
                    'debt_to_income', 'savings_balance']

print(f"{'Field':<35} {'Missing':>8} {'% Missing':>10}")
print("-" * 60)

# applicant_info fields
for field in applicant_fields:
    missing = sum(
        1 for record in data
        if record.get('applicant_info', {}).get(field) in [None, '', []]
    )
    pct = missing / len(data) * 100
    print(f"applicant_info.{field:<20} {missing:>8} {pct:>9.1f}%")

print()

# financial fields
for field in financial_fields:
    missing = sum(
        1 for record in data
        if record.get('financials', {}).get(field) is None
    )
    pct = missing / len(data) * 100
    print(f"financials.{field:<24} {missing:>8} {pct:>9.1f}%")

print()
# spending_behavior fields
empty_spending = [
    record['_id'] for record in data
    if record.get('spending_behavior') in [None, [], '']
]
print(f"Records with no spending_behavior: {len(empty_spending)}")
for app_id in empty_spending:
    print(f"  {app_id}")

# decision: check loan_approved is never missing
missing_decision = [
    record['_id'] for record in data
    if record.get('decision', {}).get('loan_approved') is None
]
print(f"Records with missing loan_approved: {len(missing_decision)}")

# logical consistency: approved but no interest rate
approved_no_rate = [
    record['_id'] for record in data
    if record.get('decision', {}).get('loan_approved') is True
    and record.get('decision', {}).get('interest_rate') is None
]
print(f"Approved loans missing interest_rate: {len(approved_no_rate)}")

# logical consistency: rejected but no rejection_reason
rejected_no_reason = [
    record['_id'] for record in data
    if record.get('decision', {}).get('loan_approved') is False
    and record.get('decision', {}).get('rejection_reason') is None
]
print(f"Rejected loans missing rejection_reason: {len(rejected_no_reason)}")

print()


Field                                Missing  % Missing
------------------------------------------------------------
applicant_info.full_name                   0       0.0%
applicant_info.email                       7       1.4%
applicant_info.ssn                         5       1.0%
applicant_info.ip_address                  5       1.0%
applicant_info.gender                      3       0.6%
applicant_info.date_of_birth               5       1.0%
applicant_info.zip_code                    2       0.4%

financials.annual_income                   5       1.0%
financials.credit_history_months           0       0.0%
financials.debt_to_income                  0       0.0%
financials.savings_balance                 0       0.0%

Records with no spending_behavior: 0
Records with missing loan_approved: 0
Approved loans missing interest_rate: 0
Rejected loans missing rejection_reason: 0



**Notes:**
- Missing values are concentrated in applicant_info fields, 
  affecting 1-1.4% of records. From a governance perspective, 
  missing SSNs and emails represent incomplete KYC checks and 
  should be flagged for follow-up before any credit decision is made.
- All decision fields are logically consistent. Every approved 
  loan has an interest rate and every rejected loan has a 
  rejection reason, meaning the decision audit trail is intact.

In [199]:
# Show which records have missing applicant_info fields
print("Records with missing applicant_info fields:\n")
for record in data:
    info   = record.get('applicant_info', {})
    blanks = [field for field in applicant_fields 
              if info.get(field) in [None, '', []]]
    if blanks:
        print(f"  {record['_id']} — missing: {blanks}")
        db.applications.update_one(
            {'_id': record['_id']},
            {'$set': {
                'data_quality_flag': 'incomplete_record',
                'missing_fields': blanks
            }}
        )

Records with missing applicant_info fields:

  app_075 — missing: ['email', 'ssn', 'ip_address', 'gender', 'date_of_birth', 'zip_code']
  app_413 — missing: ['email']
  app_120 — missing: ['email', 'ssn', 'ip_address', 'date_of_birth']
  app_268 — missing: ['email', 'ssn', 'ip_address', 'gender']
  app_377 — missing: ['email']
  app_350 — missing: ['email', 'date_of_birth']
  app_001 — missing: ['ssn', 'ip_address', 'gender', 'date_of_birth', 'zip_code']
  app_165 — missing: ['email', 'ssn', 'ip_address', 'date_of_birth']


**Note**:

Since there is no chance to fill any of these values programatically, we will flag them in MongoDB as incomplete. These require human review and further action from the applicant.

# 4. Inconsistent Data Entries

In [200]:
# Check annual_salary vs annual_income
salary_records = [
    (record['_id'], record['financials'].get('annual_salary'), record['financials'].get('annual_income'))
    for record in data
    if record.get('financials', {}).get('annual_salary')
]

print(f"Records with annual_salary field: {len(salary_records)}")
for app_id, salary, income in salary_records:
    print(f"  {app_id}: annual_salary={salary}, annual_income={income}")

Records with annual_salary field: 5
  app_436: annual_salary=45000, annual_income=None
  app_421: annual_salary=46000, annual_income=None
  app_479: annual_salary=94000, annual_income=None
  app_463: annual_salary=86000, annual_income=None
  app_449: annual_salary=75000, annual_income=None


**Notes**:
- As identified when looking at the unique fields, `annual_salary` and `annual_income` was present, likely refering to the same thing. However, there are only 5 entries with annual_salary, so we change those to annual_income for consisteny.

In [201]:
# Fix annual_salary to annual_income in MongoDB for consistency
for app_id, salary, income in salary_records:
    db.applications.update_one(
        {'_id': app_id},
        {
            '$set': {'financials.annual_income': salary},
            '$unset': {'financials.annual_salary': ''}
        }
    )
# Verify fix
remaining = db.applications.count_documents({'financials.annual_salary': {'$exists': True}})
print(f"\nRecords still with annual_salary field: {remaining}")


Records still with annual_salary field: 0


In [202]:
# Inconsistent Categorical Coding: Check all categorical fields

# Gender
gender_values = [record.get('applicant_info', {}).get('gender') for record in data]
print("Gender values:")
for val, count in Counter(gender_values).most_common():
    print(f"{repr(val):<15} {count:>5}")

print()

# Rejection reason
rejection_values = [
    record.get('decision', {}).get('rejection_reason')
    for record in data
    if record.get('decision', {}).get('loan_approved') is False
]
print("Rejection reason values:")
for val, count in Counter(rejection_values).most_common():
    print(f"{repr(val):<40} {count:>5}")

print()

# Spending categories
spending_categories = [
    item.get('category')
    for record in data
    for item in record.get('spending_behavior', [])
]
print("Spending category values:")
for val, count in Counter(spending_categories).most_common():
    print(f"{repr(val):<25} {count:>5}")

print()

# Loan purpose
loan_purposes = [
    record.get('loan_purpose')
    for record in data
    if record.get('loan_purpose') not in [None, '']
]
print("Loan purpose values:")
for val, count in Counter(loan_purposes).most_common():
    print(f"{repr(val):<25} {count:>5}")

print()

# Date of birth (detect all formats present)
def detect_date_format(raw):
    if not raw or str(raw).strip() == '': return 'missing'
    s = str(raw).strip()
    if re.match(r'^\d{4}-\d{2}-\d{2}$', s): return 'YYYY-MM-DD'
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s): return 'DD/MM/YYYY'
    if re.match(r'^\d{4}/\d{2}/\d{2}$', s): return 'YYYY/MM/DD'
    return 'unknown'

dob_formats = Counter(
    detect_date_format(record.get('applicant_info', {}).get('date_of_birth'))
    for record in data
)
print("Date of birth formats:")
for fmt, count in dob_formats.most_common():
    print(f"  {fmt:<20} {count:>5}")

Gender values:
'Male'            195
'Female'          193
'F'                58
'M'                53
''                  2
None                1

Rejection reason values:
'algorithm_risk_score'                     170
'insufficient_credit_history'               23
'high_dti_ratio'                            13
'low_income'                                 4

Spending category values:
'Travel'                     80
'Utilities'                  76
'Entertainment'              72
'Fitness'                    71
'Healthcare'                 68
'Insurance'                  68
'Dining'                     66
'Groceries'                  65
'Education'                  64
'Transportation'             61
'Rent'                       59
'Shopping'                   54
'Alcohol'                    11
'Gambling'                    7
'Adult Entertainment'         5

Loan purpose values:
'medical'                     8
'education'                   7
'vacation'                    6
'debt_consolid

**Notes**:
- `gender`: Four different representations found ('Male', 'Female', 'M', 'F') 
  plus 2 empty strings and 1 None. Will be normalised to 'Male', 'Female', 
  and 'Unknown' for missing/empty values.
- `rejection_reason`, spending categories, and loan purpose: All consistent, no normalisation needed.
- `date_of_birth`: Three different formats found: 340 records in 'YYYY-MM-DD', 
  101 in 'DD/MM/YYYY', and 56 in 'YYYY/MM/DD', plus 5 missing. 
  Will be normalised to 'YYYY-MM-DD'.

In [203]:
# Fix Gender field MongoDB
def normalize_gender(val):
    if not val or str(val).strip() == '':
        return 'Unknown'
    v = str(val).strip().upper()
    if v in ['M', 'MALE']: return 'Male'
    if v in ['F', 'FEMALE']: return 'Female'
    return 'Unknown'

for record in data:
    raw = record.get('applicant_info', {}).get('gender')
    clean = normalize_gender(raw)
    if raw != clean:
        db.applications.update_one(
            {'_id': record['_id']},
            {'$set': {'applicant_info.gender': clean}}
        )

In [204]:
# Fix date of birth to consistent format (YYYY-MM-DD) in MongoDB
from datetime import datetime

def normalize_date(raw):
    if not raw or str(raw).strip() == '':
        return None
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%Y/%m/%d'):
        try:
            return datetime.strptime(str(raw).strip(), fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return None

fixed = 0
for record in data:
    raw   = record.get('applicant_info', {}).get('date_of_birth')
    clean = normalize_date(raw)
    if raw != clean:
        db.applications.update_one(
            {'_id': record['_id']},
            {'$set': {'applicant_info.date_of_birth': clean}}
        )

# 5. Invalid or Impossible values

In [ ]:
# Convert annual_income to numeric in raw data (.json) before checks 
for record in data:
    val = record.get('financials', {}).get('annual_income')
    if isinstance(val, str):
        record['financials']['annual_income'] = int(float(val.replace(',', '').strip()))

# Verify
remaining = sum(1 for record in data if isinstance(record.get('financials', {}).get('annual_income'), str))
print("String annual_income values remaining in data list: ", remaining)

String annual_income values remaining in data list:  0


In [228]:
# SSN format (should be "XXX-XX-XXXX")
ssn_pattern = re.compile(r'^\d{3}-\d{2}-\d{4}$')
ssn_issues = [
    (record['_id'], record['applicant_info'].get('ssn'))
    for record in data
    if record.get('applicant_info', {}).get('ssn') not in [None, '']
    and not ssn_pattern.match(str(record['applicant_info'].get('ssn')))
]
print(f"SSN format issues: {len(ssn_issues)}")
for app_id, ssn in ssn_issues:
    print(f"  {app_id}: '{ssn}'")

print()

# ZIP code (should be exactly 5 digits)
zip_pattern = re.compile(r'^\d{5}$')
zip_issues = [
    (record['_id'], record['applicant_info'].get('zip_code'))
    for record in data
    if record.get('applicant_info', {}).get('zip_code') not in [None, '']
    and not zip_pattern.match(str(record['applicant_info'].get('zip_code')))
]
print(f"ZIP code format issues: {len(zip_issues)}")
for app_id, z in zip_issues:
    print(f"  {app_id}: '{z}'")

print()
# Negative or zero financial values
print("Checking financial values:\n")
financial_checks = {
    'annual_income': lambda x: x is not None and x < 0,
    'credit_history_months': lambda x: x is not None and x < 0,
    'debt_to_income': lambda x: x is not None and (x < 0 or x > 1),
    'savings_balance': lambda x: x is not None and x < 0,
}

for field, is_invalid in financial_checks.items():
    bad = [
        (record['_id'], record['financials'].get(field))
        for record in data
        if is_invalid(record.get('financials', {}).get(field))
    ]
    status = f"{len(bad)} invalid record(s)"
    print(f"{field:<30}: {status}")
    for app_id, val in bad:
        print(f"- {app_id}: {val}")

print()

# ------------------Check GDPR requirements if necessary--------------
# Age under 18 at time of application

SSN format issues: 0

ZIP code format issues: 0

Checking financial values:

annual_income                 : 0 invalid record(s)
credit_history_months         : 2 invalid record(s)
- app_043: -10
- app_156: -3
debt_to_income                : 1 invalid record(s)
- app_402: 1.85
savings_balance               : 1 invalid record(s)
- app_290: -5000



In [232]:
# credit_history_months invalid values
print("Checking credit_history_months entries:\n")
pprint(db.applications.find_one({'_id': 'app_043'}))
pprint(db.applications.find_one({'_id': 'app_156'}))
print('--'*50)
# debt_to_income invalid value
print("Checking debt_to_income entries:\n")
pprint(db.applications.find_one({'_id': 'app_402'}))
print('--'*50)
# savings_balance invalid value
print("Checking savings_balance entries:\n")
pprint(db.applications.find_one({'_id': 'app_290'}))

Checking credit_history_months entries:

{'_id': 'app_043',
 'applicant_info': {'date_of_birth': '1985-08-17',
                    'email': 'daniel.king27@mail.com',
                    'full_name': 'Daniel King',
                    'gender': 'Male',
                    'ip_address': '10.204.248.60',
                    'ssn': '166-23-2000',
                    'zip_code': '10038'},
 'decision': {'approved_amount': 68000,
              'interest_rate': 3.9,
              'loan_approved': True},
 'financials': {'annual_income': 131000,
                'credit_history_months': -10,
                'debt_to_income': 0.06,
                'savings_balance': 53098},
 'spending_behavior': [{'amount': 398, 'category': 'Groceries'},
                       {'amount': 464, 'category': 'Entertainment'}]}
{'_id': 'app_156',
 'applicant_info': {'date_of_birth': '1999-11-28',
                    'email': 'jessica.green86@gmail.com',
                    'full_name': 'Jessica Green',
                

**Notes**:
- `credit_history_months`: 2 records with negative values. Credit history cannot be negative.
  - app_043: Was approved for $68,000 at 3.9% interest. Algorithm ignored an impossible value and approved anyway.
  - app_156: Was rejected, but for algorithm_risk_score, not for invalid credit history.
- `debt_to_income`: 1 record with a value of 1.85. While a DTI 
  above 1.0 is uncommon but possible, this value is extreme enough to suggest an  entry error. This is further supported by the fact that the applicant 
  has an annual income of $88,000 and a savings balance of $37,281. Yet she was 
  approved for a loan despite the impossible DTI value, suggesting the 
  algorithm failed to validate this field before making a credit decision. 
- `savings_balance`: 1 record with negative value. Could indicate an overdraft but requires manual verification.
- All 4 records flagged for human review.

In [234]:
# Flag all invalid financial value records in MongoDB
invalid_records = [
    ('app_043', 'invalid_credit_history_months', 'Negative value: -10'),
    ('app_156', 'invalid_credit_history_months', 'Negative value: -3'),
    ('app_402', 'invalid_debt_to_income', 'Value of 1.85, likely decimal error (0.185). Approved despite impossible DTI.'),
    ('app_290', 'invalid_savings_balance', 'Negative value: -5000. Approved despite negative savings balance.'),
]

for app_id, flag, note in invalid_records:
    db.applications.update_one(
        {'_id': app_id},
        {'$set': {
            'data_quality_flag': flag,
            'flag_note': note
        }}
    )

# 6. Pipeline Code

In [238]:
# Export a flat .csv for analysis, plus separate file for spending_behavior
# Pull clean data from MongoDB
records = list(db.applications.find({}))

# Main flat DataFrame
df = pd.json_normalize(records, sep='_')
df.rename(columns={'_id': 'app_id'}, inplace=True)

# Drop the raw nested columns
df = df.drop(columns=[col for col in ['applicant_info', 'financials', 
                                   'decision', 'spending_behavior'] 
                       if col in df.columns])

print(df.columns.tolist())
print(f"Shape: {df.shape}")
df.to_csv('data/processed/applications_clean.csv', index=False)

# Separate spending_behavior file
spending_rows = []
for record in records:
    for item in record.get('spending_behavior', []):
        spending_rows.append({
            'app_id'  : record['_id'],
            'category': item.get('category'),
            'amount'  : item.get('amount')
        })

df_spending = pd.DataFrame(spending_rows)
df_spending.to_csv('data/processed/spending_behavior.csv', index=False)

['app_id', 'processing_timestamp', 'applicant_info_full_name', 'applicant_info_email', 'applicant_info_ssn', 'applicant_info_ip_address', 'applicant_info_gender', 'applicant_info_date_of_birth', 'applicant_info_zip_code', 'financials_annual_income', 'financials_credit_history_months', 'financials_debt_to_income', 'financials_savings_balance', 'decision_loan_approved', 'decision_rejection_reason', 'loan_purpose', 'decision_interest_rate', 'decision_approved_amount', 'notes', 'data_quality_flag', 'missing_fields', 'flag_note']
Shape: (500, 22)
